In [1]:
import boto3
import os
from dotenv import load_dotenv
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
import io


In [2]:
# Load AWS Keys
load_dotenv()
aws_key_id = os.environ.get('aws_key_id')
aws_secret_key = os.environ.get('aws_secret_key')

In [3]:
# Load CSV from S3
s3 = boto3.client(
    's3', 
    aws_access_key_id=aws_key_id,
    aws_secret_access_key=aws_secret_key,
)
obj = s3.get_object(Bucket='endgbv', Key='raw_data/ENDGBV.csv')
df = pd.read_csv(io.BytesIO(obj['Body'].read()))


C:\Users\helen\AppData\Local\Temp\ipykernel_11228\2670726172.py:8: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.BytesIO(obj['Body'].read()))


In [4]:
# Load Username and PW for RDS
user = os.environ.get('username')
pw = os.environ.get('password')

In [6]:
# Connect to RDS
engine = create_engine(
    'postgresql://' + user + ":" + pw + 
    "@endgbv-database.cx62ou6gkd87.us-east-2.rds.amazonaws.com" + 
    ":5432/postgres?sslmode=require"
)


In [7]:
# Upload to RDS
df.to_sql('endgbv', engine, if_exists='replace', index=False)
print("Done! Rows loaded:", len(df))

Done! Rows loaded: 483790
